# TravelMate AI — Giai đoạn 3: Đánh giá

Notebook sinh phản hồi trên tập Test và chấm các hành vi quan trọng của TravelMate. Kết quả này nên được bổ sung bằng đánh giá thủ công về độ hữu ích và tính đúng đắn của lịch trình.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/trongnd16092005/travelmate-ai.git"
BRANCH = "feature/ai-itinerary-generation"
REPO_DIR = Path("/content/travelmate-ai")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR / "services" / "ai-service")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[training]"], check=True)

In [ ]:
SOURCE_DATASET = Path("training/data/travelmate_train.sample.jsonl")
PROCESSED_DIR = Path("training/data/processed")
subprocess.run(
    [
        sys.executable,
        "-m",
        "training.prepare_dataset",
        str(SOURCE_DATASET),
        "--output-dir",
        str(PROCESSED_DIR),
        "--seed",
        "42",
    ],
    check=True,
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ADAPTER_PATH = Path("/content/drive/MyDrive/TravelMate/artifacts/travelmate-qwen3-4b-lora")
PREDICTIONS_PATH = Path("training/outputs/test_predictions.jsonl")
REPORT_PATH = Path("training/outputs/evaluation_report.json")

if not ADAPTER_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy adapter tại {ADAPTER_PATH}. Hãy chạy notebook 02 trước."
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "training.generate_predictions",
        "--dataset",
        str(PROCESSED_DIR / "test.jsonl"),
        "--adapter-path",
        str(ADAPTER_PATH),
        "--output",
        str(PREDICTIONS_PATH),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "training.evaluate_predictions",
        "--dataset",
        str(PROCESSED_DIR / "test.jsonl"),
        "--predictions",
        str(PREDICTIONS_PATH),
        "--output",
        str(REPORT_PATH),
    ],
    check=True,
)

In [ ]:
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
print(json.dumps(report, ensure_ascii=False, indent=2))
print("\nHãy đọc thêm từng phản hồi trong:", PREDICTIONS_PATH)